# Polarizability using automatic differentiation

Simple example for computing properties using (forward-mode)
automatic differentiation.
For a more classical approach and more details about computing polarizabilities,
see Polarizability by linear response.

In [1]:
using DFTK
using LinearAlgebra
using ForwardDiff
using PseudoPotentialData

# Construct PlaneWaveBasis given a particular electric field strength
# Again we take the example of a Helium atom.
function make_basis(ε::T; a=10., Ecut=30) where {T}
    lattice = T(a) * I(3)  # lattice is a cube of $a$ Bohrs
    # Helium at the center of the box
    pseudopotentials = PseudoFamily("cp2k.nc.sr.lda.v0_1.semicore.gth")
    atoms     = [ElementPsp(:He, pseudopotentials)]
    positions = [[1/2, 1/2, 1/2]]

    model = model_DFT(lattice, atoms, positions;
                      functionals=[:lda_x, :lda_c_vwn],
                      extra_terms=[ExternalFromReal(r -> -ε * (r[1] - a/2))],
                      symmetries=false)
    PlaneWaveBasis(model; Ecut, kgrid=[1, 1, 1])  # No k-point sampling on isolated system
end

# dipole moment of a given density (assuming the current geometry)
function dipole(basis, ρ)
    @assert isdiag(basis.model.lattice)
    a  = basis.model.lattice[1, 1]
    rr = [a * (r[1] - 1/2) for r in r_vectors(basis)]
    sum(rr .* ρ) * basis.dvol
end

# Function to compute the dipole for a given field strength
function compute_dipole(ε; tol=1e-8, kwargs...)
    scfres = self_consistent_field(make_basis(ε; kwargs...); tol)
    dipole(scfres.basis, scfres.ρ)
end;

With this in place we can compute the polarizability from finite differences
(just like in the previous example):

In [2]:
polarizability_fd = let
    ε = 0.01
    (compute_dipole(ε) - compute_dipole(0.0)) / ε
end

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -2.770816729884                   -0.53    8.0    154ms
  2   -2.772140416991       -2.88       -1.31    1.0    121ms
  3   -2.772170185461       -4.53       -2.54    1.0    101ms
  4   -2.772170661937       -6.32       -3.41    1.0    119ms
  5   -2.772170721845       -7.22       -3.97    2.0    122ms
  6   -2.772170722974       -8.95       -4.97    1.0    107ms
  7   -2.772170723012      -10.42       -5.32    1.0    114ms
  8   -2.772170723015      -11.62       -5.74    2.0    131ms
  9   -2.772170723015      -12.54       -6.71    1.0    116ms
 10   -2.772170723015      -13.82       -7.58    1.0    116ms
 11   -2.772170723015      -14.35       -7.97    2.0    131ms
 12   -2.772170723015   +  -13.70       -8.62    1.0    116ms
n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -2.7

1.7735580785834286

We do the same thing using automatic differentiation. Under the hood this uses
custom rules to implicitly differentiate through the self-consistent
field fixed-point problem. This leads to a density-functional perturbation
theory problem, which is automatically set up and solved in the background.

In [3]:
polarizability = ForwardDiff.derivative(compute_dipole, 0.0)
println()
println("Polarizability via ForwardDiff:       $polarizability")
println("Polarizability via finite difference: $polarizability_fd")

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -2.770779832702                   -0.52    9.0    220ms
  2   -2.772060627913       -2.89       -1.32    1.0    122ms
  3   -2.772083084445       -4.65       -2.45    1.0    102ms
  4   -2.772083340360       -6.59       -3.15    1.0    109ms
  5   -2.772083416617       -7.12       -3.97    2.0    123ms
  6   -2.772083417547       -9.03       -4.35    1.0    112ms
  7   -2.772083417808       -9.58       -5.36    1.0    114ms
  8   -2.772083417810      -11.62       -5.74    1.0    117ms
  9   -2.772083417811      -12.73       -6.22    1.0    117ms
 10   -2.772083417811      -13.27       -7.27    2.0    320ms
 11   -2.772083417811      -13.85       -8.02    1.0    131ms
Solving response problem
Iter  Restart  Krydim  log10(res)  avg(CG)  Δtime   Comment
----  -------  ------  ----------  -------  ------  ---------------
                                      13.0